# Test full pipeline: import -> graph -> chat Q&A -> checklist evaluation

Khác với `import_flow.ipynb` (chỉ dừng ở bước build graph), notebook này chạy tiếp 2 luồng sử dụng thực tế của app:

1. Import + build graph (rebuild từ file `.md` mới nhất - đã dọn khoảng trắng thừa)
2. Hỏi đáp tự do (`llm/chat_qa.py`) với vài câu hỏi mẫu
3. Đánh giá checklist qua LangGraph (`rag/checklist_graph.py`) với 1 checklist nhỏ nhiều category
4. Tổng hợp kết quả checklist thành bảng

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

CONTRACT_ID = 1
CONTRACT_PATH = ROOT / "data" / "input" / "BM 01 - Hop dong cung cap dich vu phat trien phan mem.md"

## 1. Import + build graph

In [ ]:
from ingestion.section_splitter import split_into_sections_and_clauses
from knowledge_graph.graph_extraction import extract_relations
from knowledge_graph.builder import build_graph, clear_graph

markdown_text = CONTRACT_PATH.read_text(encoding="utf-8")
contract_name = CONTRACT_PATH.stem

sections, clauses = split_into_sections_and_clauses(markdown_text)
print(f"{len(sections)} Phan, {len(clauses)} Dieu")

extraction, extraction_usage = extract_relations(contract_name, clauses)
print(f"Parties: {[p.name for p in extraction.parties]}")

clear_graph()
n_cross = build_graph(CONTRACT_ID, contract_name, sections, clauses, extraction, extraction_usage)
print(f"Da build {len(clauses)} Clause, {n_cross} quan he cheo, + embedding.")

In [ ]:
# Xac nhan text da duoc don khoang trang thua (khong con "Nhan :\n\nGia tri" tach dong)
clause_0 = next(c for c in clauses if c["number"] == "0")
print(clause_0["text"][:600])

## 2. Hỏi đáp tự do (chat Q&A)

Mỗi câu hỏi độc lập, không giữ ngữ cảnh - đúng như luồng chat trong `app.py`. Từ khi hợp nhất, chat KHÔNG còn dùng module riêng nữa - chỉ là gọi `rag/checklist_graph.py` với `pass_criteria`/`violation_criteria` để trống (xem TRƯỜNG HỢP ĐẶC BIỆT trong `prompts/checklist_evaluation_system_prompt.txt`).

In [ ]:
from rag.checklist_graph import evaluate_checklist_item

SAMPLE_QUESTIONS = [
    "Người đại diện ký hợp đồng đã xác định đúng theo phân cấp, ủy quyền chưa?",
    "Mức phạt vi phạm hợp đồng là bao nhiêu?",
    "Hợp đồng có điều khoản bảo mật thông tin không?",
]

for q in SAMPLE_QUESTIONS:
    item = {"id": "chat", "category": "", "question": q, "pass_criteria": "", "violation_criteria": "", "note": "", "severity": "medium", "needs_search": False}
    evaluation, found, usage = evaluate_checklist_item(CONTRACT_ID, item)
    print(f"### Cau hoi: {q}")
    print(f"status={evaluation.status} confidence={evaluation.confidence} usage={usage}")
    print(evaluation.evidence)
    print()
    print("=" * 80)
    print()

## 3. Đánh giá checklist qua LangGraph

`rag/checklist_graph.py`: mỗi mục chạy graph 2 node `retrieve -> evaluate` (LangGraph `StateGraph`), dùng prompt nghiêm ngặt hơn chat (`checklist_evaluation_system_prompt.txt` - buộc kết luận pass/fail dứt khoát, ưu tiên fail khi còn nghi ngờ).

Checklist mẫu nhỏ, nhiều category khác nhau để test bao quát (không cần file JSON, viết inline).

In [ ]:
SAMPLE_CHECKLIST = [
    {
        "id": "dai_dien_uy_quyen",
        "category": "Ủy quyền ký kết",
        "question": "Người đại diện ký hợp đồng đã xác định đúng theo phân cấp, ủy quyền ký kết chưa?",
        "pass_criteria": "Người ký là Giám đốc/Tổng Giám đốc hoặc có giấy ủy quyền hợp lệ: đúng phạm vi, còn hiệu lực tại ngày ký.",
        "violation_criteria": "- Người ký không có thẩm quyền, không có ủy quyền\n- Phó Giám đốc hoặc Người khác ký hợp đồng giá trị trên 2 tỷ đồng.\n- Ngày giấy ủy quyền sau ngày ký hợp đồng",
        "note": "Các hợp đồng có giá trị trên 2 tỷ đồng phải do Giám đốc ký",
        "severity": "high",
        "needs_search": False,
    },
    {
        "id": "phat_vi_pham",
        "category": "Phạt, bồi thường",
        "question": "Điều khoản phạt vi phạm có quy định rõ mức phạt và giới hạn tối đa không?",
        "pass_criteria": "Có quy định rõ mức phạt (%) và tổng mức phạt tối đa không vượt quá 8% giá trị phần nghĩa vụ vi phạm theo quy định pháp luật.",
        "violation_criteria": "Không quy định mức phạt, hoặc mức phạt tối đa vượt quá 8% giá trị nghĩa vụ vi phạm.",
        "note": "",
        "severity": "high",
        "needs_search": False,
    },
    {
        "id": "bao_hanh",
        "category": "Bảo hành",
        "question": "Hợp đồng có quy định thời hạn bảo hành sản phẩm/dịch vụ không?",
        "pass_criteria": "Có quy định rõ thời hạn bảo hành (số tháng cụ thể) và phạm vi bảo hành.",
        "violation_criteria": "Không có điều khoản bảo hành, hoặc không nêu rõ thời hạn.",
        "note": "",
        "severity": "medium",
        "needs_search": False,
    },
    {
        "id": "bao_hiem_khong_co",
        "category": "Bảo hiểm",
        "question": "Hợp đồng có yêu cầu VTIT mua bảo hiểm trách nhiệm nghề nghiệp không?",
        "pass_criteria": "Có điều khoản yêu cầu VTIT mua bảo hiểm trách nhiệm nghề nghiệp trong suốt thời gian thực hiện hợp đồng.",
        "violation_criteria": "Không có điều khoản nào về bảo hiểm.",
        "note": "Mục này để test trường hợp hợp đồng THỰC SỰ không có điều khoản liên quan (kỳ vọng fail rõ ràng).",
        "severity": "low",
        "needs_search": False,
    },
]

print(f"{len(SAMPLE_CHECKLIST)} muc checklist mau, {len(set(i['category'] for i in SAMPLE_CHECKLIST))} category.")

In [ ]:
import concurrent.futures
from rag.checklist_graph import evaluate_checklist_item

results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(evaluate_checklist_item, CONTRACT_ID, item): item for item in SAMPLE_CHECKLIST}
    for future in concurrent.futures.as_completed(futures):
        item = futures[future]
        evaluation, cited_clauses, usage = future.result()
        results.append({"item": item, "evaluation": evaluation, "cited_clauses": cited_clauses, "usage": usage})

print(f"Da danh gia {len(results)}/{len(SAMPLE_CHECKLIST)} muc.")

## 4. Tổng hợp kết quả

In [ ]:
n_pass = sum(1 for r in results if r["evaluation"].status == "pass")
n_fail = len(results) - n_pass
print(f"Tong: {len(results)} muc - {n_pass} PASS, {n_fail} FAIL\n")

for r in sorted(results, key=lambda r: (r["evaluation"].status != "fail", r["item"]["category"])):
    item = r["item"]
    ev = r["evaluation"]
    cited = ", ".join(f"D{c['number']}" for c in r["cited_clauses"])
    print(f"[{ev.status.upper():4}] [{item['category']}] {item['question']}")
    print(f"  Cited: {cited}")
    print(f"  Evidence: {ev.evidence[:200]}...")
    print(f"  Proposal: {ev.proposal[:200]}...")
    print()

In [ ]:
# Kiem tra nhanh: muc "bao_hiem_khong_co" (that su khong co dieu khoan lien quan) co bi FAIL dut khoat khong,
# khong duoc "huong loi tu su nghi ngo" (benefit of the doubt).
no_insurance_result = next(r for r in results if r["item"]["id"] == "bao_hiem_khong_co")
assert no_insurance_result["evaluation"].status == "fail", "Ky vong FAIL vi hop dong khong co dieu khoan bao hiem"
print("OK: muc khong co dieu khoan lien quan duoc dut khoat danh gia FAIL, khong hedge.")